# W13 · 분할과 누수 (노트북 01 · 약 5분)

**무엇을 하나** — COMID MD 336행(AAA1 · 노화 3 × 온도 18 × 반복 약 6)으로 같은 모델 4종을 분할 4가지로 평가한다.

**왜 하나** — 같은 조건의 반복(시드만 다른 쌍둥이)이 학습과 시험에 나뉘면 점수가 부풀려진다. 분할 방식 = 평가 질문.

**실행** — 기본 = week13 폴더의 Claude Code 에 「`@notebooks/01_split_and_leak.ipynb` 를 실행하고 결과를 요약해」 → week13 폴더에서 `python ../run_nb.py notebooks/01_split_and_leak.ipynb` (출력이 이 파일에 저장 · Claude Code 가 직접 읽음). 직접 보려면 `jupyter lab` 으로 열어 셀을 위에서 아래로 실행한다. 덱 7장(P13-1) = 셀 1 출력 · 13장(P13-2) = 셀 2 · 12장(바꿔 보기) = 셀 3.

In [ ]:
# 0) 작업 폴더 맞추기 — 로컬 · Claude Code 가 기본 · Colab 은 선택
#    run_nb.py 는 week13 에서 시작(그대로) · JupyterLab 은 notebooks/ 에서 시작 → week13 으로 이동 · Colab 이면 저장소를 받는다
#    scikit-learn 은 mdai 에 포함(../../environment.yml)
import os, pathlib
if not pathlib.Path('scripts/common.py').exists():
    if pathlib.Path('../scripts/common.py').exists():
        os.chdir('..')                       # 로컬: notebooks/ → week13
    else:                                     # Colab(선택): 저장소 받기
        !git clone https://github.com/sentirain7/2026_MOOC_Lecture_Materials.git repo
        %cd repo/week13
print(os.getcwd())

In [ ]:
# 1) 데이터 요약 — 한 행 · 그룹 · 표적 · MD 반복 오차 (P13-1 · Claude Code 가 이 출력을 읽는다)
import sys; sys.path.insert(0, 'scripts')
from common import load
d, X, y = load('density')
print('행', len(d), '· 셀', d.cell.nunique(), '· 온도', d.T_K.nunique(), f'({d.T_K.min()}~{d.T_K.max()} K)', '· 조건(셀@온도)', d.cond.nunique())
print('조건당 반복 수:', d.groupby('cond').size().value_counts().sort_index().to_dict())
for t, u in [('density', 'g/cm3'), ('ced', 'MJ/m3'), ('bulk_modulus', 'GPa')]:
    sd = d.groupby('cond')[t].std().median()
    print(f'{t:13s} 범위 {d[t].min():.4g} ~ {d[t].max():.4g} {u} · 반복 간 SD 중앙값 {sd:.4g}')
d.head(4)

In [ ]:
# 2) 분할 4종 × 모델 4종 — scripts/split_compare.py (약 1분 · CPU) → out/split_compare.json
import subprocess, json, pandas as pd
subprocess.run([sys.executable, 'scripts/split_compare.py'], check=True)
r = json.load(open('out/split_compare.json'))
rows = [{'분할': s, '누수': v['leak_frac'], '모델': m, 'RMSE': x['rmse'], 'R2': x['r2'], 'n': v['n_test']}
        for s, v in r['splits'].items() for m, x in v['models'].items()]
pd.DataFrame(rows).pivot(index='모델', columns='분할', values='RMSE')[['random', 'cond', 'loago', 't_extrap']]

In [ ]:
# 3) 바꿔 보기 (덱 12장) — SPLIT 만 'random' ↔ 'loago' 로 바꿔 ▶ · 숲의 학습 · 시험 RMSE 배율
SPLIT = 'random'          # 'random' (행 무작위 80/20) 또는 'loago' (노화 상태 1종 빼기)
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ShuffleSplit
from common import metrics, SEED
idx = np.arange(len(d))
folds = list(ShuffleSplit(1, test_size=0.2, random_state=SEED).split(idx)) if SPLIT == 'random' \
        else [(idx[d.aging != k], idx[d.aging == k]) for k in (0, 1, 2)]
tr_e, te_e = [], []
for tr, te in folds:
    m = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=1).fit(X[tr], y[tr])
    tr_e.append(metrics(y[tr], m.predict(X[tr]))['rmse']); te_e.append(metrics(y[te], m.predict(X[te]))['rmse'])
a, b = np.mean(tr_e), np.mean(te_e)
print(f'{SPLIT}: 학습 RMSE {a:.4f} · 시험 RMSE {b:.4f} · 시험 ÷ 학습 = {b/a:.1f}배 · MD 반복 오차 0.0027')

## 확인
- 셀 1: 행 336 · 셀 3 · 온도 18 · 조건 54 · 밀도 반복 SD 중앙값 약 0.0027
- 셀 2: 숲 RMSE random 0.0036 → cond 0.0062 → loago 0.0299 · 선형 loago 0.0077 (`expected_output/split_compare.json` 과 같은지)
- 셀 3: random 은 배율 약 1.2 · loago 는 약 10 — 배율이 1 에 가깝고 R² 가 0.98 이상이면 먼저 누수를 의심

못 돌아가면 `expected_output/split_compare.json` 으로 P13-2 를 한다.